# Introducción a la Ingeniería de Prompts
La ingeniería de prompts es el proceso de diseñar y optimizar prompts para tareas de procesamiento de lenguaje natural. Implica seleccionar los prompts adecuados, ajustar sus parámetros y evaluar su rendimiento. La ingeniería de prompts es crucial para lograr una alta precisión y eficiencia en los modelos de PLN. En esta sección, exploraremos los conceptos básicos de la ingeniería de prompts usando los modelos de OpenAI para la exploración.


### Ejercicio 1: Tokenización
Explora la tokenización usando tiktoken, un tokenizador rápido de código abierto de OpenAI
Consulta [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) para más ejemplos.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Ejercicio 2: Validar la configuración de la clave de Microsoft Foundry Models

> **Nota:** GitHub Models se retirará a finales de julio de 2026. Este ejercicio utiliza [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) en su lugar, que ofrece el mismo catálogo de modelos gratuitos para probar y la experiencia del SDK de Inferencia de Azure AI.

Ejecuta el código a continuación para verificar que tu punto de conexión de Microsoft Foundry Models esté configurado correctamente. El código simplemente intenta un prompt básico simple y valida la finalización. La entrada `oh say can you see` debería completarse más o menos como `by the dawn's early light..`


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

# temperature/top_p need a non-reasoning model (gpt-5 rejects them), so use a Llama model here.
model_name = os.environ.get("AZURE_INFERENCE_CHAT_MODEL", "Llama-3.3-70B-Instruct")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### Ejercicio 3: Fabricaciones
Explora qué sucede cuando le pides al LLM que devuelva completaciones para un mensaje sobre un tema que puede no existir, o sobre temas que quizá no conoce porque estaban fuera de su conjunto de datos preentrenado (más recientes). Observa cómo cambia la respuesta si intentas un mensaje diferente, o un modelo distinto.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### Ejercicio 4: Basado en instrucciones 
Utiliza la variable "text" para establecer el contenido principal 
y la variable "prompt" para proporcionar una instrucción relacionada con ese contenido principal.

Aquí pedimos al modelo que resuma el texto para un estudiante de segundo grado


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### Ejercicio 5: Prompt Complejo 
Prueba una solicitud que tenga mensajes de sistema, usuario y asistente 
El sistema establece el contexto del asistente
Los mensajes de usuario y asistente proporcionan contexto de conversación de múltiples turnos

Observa cómo la personalidad del asistente se establece como "sarcástica" en el contexto del sistema. 
Intenta usar un contexto de personalidad diferente. O prueba una serie diferente de mensajes de entrada/salida


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### Ejercicio: Explora Tu Intuición
Los ejemplos anteriores te ofrecen patrones que puedes usar para crear nuevos prompts (simples, complejos, instrucciones, etc.) - intenta crear otros ejercicios para explorar algunas de las otras ideas de las que hemos hablado, como ejemplos, pistas y más.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
